In [7]:
# Input take label, latent semantics and k
LABEL_INPUT = input(
"""
Provide the label for which you want to find similar labels
"""
)

LATENT_SEMANTICS = input(
"""
Provide a latent semantic that was generated using tasks 3 to 6.
For eg:
if you generated a latent semantic in task 3 from color space and svd,
enter "LS1_color_svd" if you generated it for tasks where you didn't select the algorithm,
use this format "LS3_color"
If you are unsure what to type here, please see Code/database/<name>_reducer.pt
after running the relevant task. Any <name> can be input here.
"""
)

K = int(input("Enter K, the K most similar labels to find under the latent space."))

In [12]:
from utils.database_utils import retrieve
from utils.distance_utils import top_k_distance_ranker
from scipy.spatial.distance import cityblock, correlation, cosine
from utils.dataset_utils import initialize_dataset

from feature_models.feature_matrix.label_label_similarity import LabelLabelSimilarity

In [9]:
LATENT_SPACE = LATENT_SEMANTICS.split('_')[0]
FEATURE_SPACE = LATENT_SEMANTICS.split('_')[1]

# Load created latent space
reducer = retrieve(f'{LATENT_SEMANTICS}_reducer.pt')

feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

In [10]:
def find_nearest_cluster(centroids, query_vector, K, distance_fn = cityblock):
    distance = []
    for i, label in enumerate(centroids.keys()):
        distance.append((label, distance_fn(query_vector, centroids[label])))
    distance.sort(key=lambda x:x[1])
    return distance[:K]

In [13]:
if LATENT_SPACE == "LS1" or LATENT_SPACE == "LS4":
    # For LS1, LS2, LS4, find labels for every image in the latent space
    latent_space = reducer.reduce_features(feature_vectors)
    centroid_format_latent_space = {}
    for i, feature_item in enumerate(feature_vectors.items()):
        label = feature_item[1][0]
        feature = latent_space[i]
        centroid_format_latent_space[i] = (label, feature)

    # Feed it to the centroid function
    from utils.vector_utils import get_representative_vectors_for_labels
    from utils.dataset_utils import initialize_dataset

    centroids = get_representative_vectors_for_labels(centroid_format_latent_space, initialize_dataset().categories, 1)

    # Find closest labels based on distance
    distances = find_nearest_cluster(centroids, centroids[LABEL_INPUT], K)
    print(distances)
if LATENT_SPACE == "LS2":
    # For LS2 we pick the matrix from CP decomposition
    labels = initialize_dataset().categories
    latent_space = reducer.label_latent_space_weights
    centroids = {}
    # Make latent space in label format
    for i, label in enumerate(labels):
        centroids[label] = latent_space[i]
    # Find closest labels based on distance
    distances = find_nearest_cluster(centroids, centroids[LABEL_INPUT], K)
    print(distances)
else:
    # For LS3 
    # Find distance directly
    formatter = LabelLabelSimilarity(feature_vectors)
    label_feature_vectors = formatter.get_matrix()
    latent_space = reducer.reduce_features(label_feature_vectors)
    labels = initialize_dataset().categories
    centroids = {}
    # Make latent space in label format
    for i, label in enumerate(labels):
        centroids[label] = latent_space[i]
    # Find closest labels based on distance
    distances = find_nearest_cluster(centroids, centroids[LABEL_INPUT], K)
    print(distances)

Files already downloaded and verified
[('airplanes', 0.0), ('yin_yang', 2.0013565896373446), ('wheelchair', 2.0013565911412203), ('scissors', 2.0013565911643543), ('cellphone', 2.001356591164701), ('menorah', 2.0013565911647224), ('butterfly', 2.0013565911647246), ('Leopards', 2.001356591164727), ('accordion', 2.001356591164727), ('anchor', 2.001356591164727)]


In [15]:
for index, label_weights in enumerate(distances):
    label = label_weights[0]
    distance = label_weights[1]
    print(str(index + 1) + ".", "\t\tLabel: ", label, "\t\tDistance: ", distance)


1. 		Label:  airplanes 		Distance:  0.0
2. 		Label:  yin_yang 		Distance:  2.0013565896373446
3. 		Label:  wheelchair 		Distance:  2.0013565911412203
4. 		Label:  scissors 		Distance:  2.0013565911643543
5. 		Label:  cellphone 		Distance:  2.001356591164701
6. 		Label:  menorah 		Distance:  2.0013565911647224
7. 		Label:  butterfly 		Distance:  2.0013565911647246
8. 		Label:  Leopards 		Distance:  2.001356591164727
9. 		Label:  accordion 		Distance:  2.001356591164727
10. 		Label:  anchor 		Distance:  2.001356591164727
